**Install PyGithub to access Github Rest API**

In [17]:
!pip install PyGithub

**Import necessary python modules**

In [18]:
from github import Github, Auth
import csv
import logging
from datetime import date, timedelta
from google.colab import files

logging.getLogger("github").setLevel(logging.WARNING)

**Authorization**

In [ ]:
# GitHub Personal Access Token
GITHUB_TOKEN = ""

# Use new Auth method
auth = Auth.Token(GITHUB_TOKEN)

# Initialize GitHub client with the new auth object
g = Github(auth=auth)

**Apply Search Query with Filters and Slicing Period**

In [20]:
def generate_semi_year_slices(start_date, end_date):
    """
    Generate 6-month (semiannual) date ranges.
    """
    start = date.fromisoformat(start_date)
    end = date.fromisoformat(end_date)

    slices = []
    current = start

    while current <= end:
        if current.month <= 6:
            slice_start = date(current.year, 1, 1)
            slice_end = date(current.year, 6, 30)
            next_start = date(current.year, 7, 1)
        else:
            slice_start = date(current.year, 7, 1)
            slice_end = date(current.year, 12, 31)
            next_start = date(current.year + 1, 1, 1)

        if slice_start < start:
            slice_start = start
        if slice_end > end:
            slice_end = end

        slices.append((slice_start.isoformat(), slice_end.isoformat()))
        current = next_start

    return slices

In [21]:
def generate_quarter_slices(start_date, end_date):
    """
    Generate 3-month (quarter) date ranges.
    """
    start = date.fromisoformat(start_date)
    end = date.fromisoformat(end_date)

    slices = []
    current = start

    while current <= end:
        quarter = (current.month - 1) // 3
        q_start_month = quarter * 3 + 1
        q_end_month = q_start_month + 2

        q_start = date(current.year, q_start_month, 1)

        # compute end of quarter
        if q_end_month == 12:
            q_end = date(current.year, 12, 31)
            next_start = date(current.year + 1, 1, 1)
        else:
            next_start = date(current.year, q_end_month + 1, 1)
            q_end = next_start - timedelta(days=1)

        # clip to bounds
        if q_start < start:
            q_start = start
        if q_end > end:
            q_end = end

        slices.append((q_start.isoformat(), q_end.isoformat()))
        current = next_start

    return slices

In [22]:
def generate_monthly_slices(start_date, end_date):
    """
    Generate monthly date ranges.
    """
    start = date.fromisoformat(start_date)
    end = date.fromisoformat(end_date)

    slices = []
    current = start.replace(day=1)

    while current <= end:
        # first day of next month
        if current.month == 12:
            next_month = date(current.year + 1, 1, 1)
        else:
            next_month = date(current.year, current.month + 1, 1)

        month_start = current
        month_end = next_month - timedelta(days=1)

        # clip to bounds
        if month_start < start:
            month_start = start
        if month_end > end:
            month_end = end

        slices.append((month_start.isoformat(), month_end.isoformat()))
        current = next_month

    return slices

In [23]:
def generate_partialmonth_slices(start_date, end_date):
    """
    Generate 15-day date ranges.
    """
    start = date.fromisoformat(start_date)
    end = date.fromisoformat(end_date)

    slices = []
    current = start

    while current <= end:
        slice_start = current
        slice_end = min(current + timedelta(days=14), end)

        slices.append((
            slice_start.isoformat(),
            slice_end.isoformat()
        ))

        current = slice_end + timedelta(days=1)

    return slices

**Mine Github Repositories**

In [24]:
llm_keywords = ["LLM", '"large language model"']
agent_keywords = ["agent", "autonomous", '"multi-agent"', '"task automation"']

base_filters = "language:Python fork:false archived:false"


In [25]:
def mine_repositories(g, llm_keywords, agent_keywords, base_filters, start_year=2022, end_year=2025):
    """
    Mine GitHub repositories with adaptive time slicing.

    Returns:
        dict: {repo.full_name: repo}
    """
    all_repos = {}

    for llm in llm_keywords:
        for agent in agent_keywords:
          for year in range(start_year, end_year + 1):
            y_start = f"{year}-01-01"
            y_end = f"{year}-12-31"

            year_query = f'{llm} {agent} {base_filters} created:{y_start}..{y_end}'
            year_results = g.search_repositories(query=year_query)
            print("\nRunning query:", year_query)
            print(f"Year slice {year}: {year_results.totalCount}")

            if year_results.totalCount < 1000:
                for repo in year_results:
                    all_repos[repo.full_name] = repo
                continue

            # Semi-year slicing
            for s_start, s_end in generate_semi_year_slices(y_start, y_end):
                semi_query = f'{llm} {agent} {base_filters} created:{s_start}..{s_end}'
                semi_results = g.search_repositories(query=semi_query)

                print(f"Semi-yearly slice {s_start}..{s_end}: {semi_results.totalCount}")

                if semi_results.totalCount < 1000:
                    for repo in semi_results:
                        all_repos[repo.full_name] = repo
                    continue

                # Quarter slicing
                for q_start, q_end in generate_quarter_slices(s_start, s_end):
                    quarter_query = f'{llm} {agent} {base_filters} created:{q_start}..{q_end}'
                    quarter_results = g.search_repositories(query=quarter_query)

                    print(f"Quarterly slice {q_start}..{q_end}: {quarter_results.totalCount}")

                    if quarter_results.totalCount < 1000:
                        for repo in quarter_results:
                            all_repos[repo.full_name] = repo
                        continue

                    # Monthly slicing
                    for m_start, m_end in generate_monthly_slices(q_start, q_end):
                        month_query = f'{llm} {agent} {base_filters} created:{m_start}..{m_end}'
                        month_results = g.search_repositories(query=month_query)

                        print(f"Monthly slice {m_start}..{m_end}: {month_results.totalCount}")

                        if month_results.totalCount < 1000:
                            for repo in month_results:
                                all_repos[repo.full_name] = repo
                            continue

                        # 15-DAY slicing
                        for f_start, f_end in generate_partialmonth_slices(m_start, m_end):
                            partialmonth_query = f'{llm} {agent} {base_filters} created:{f_start}..{f_end}'
                            partialmonth_results = g.search_repositories(query=partialmonth_query)

                            print(f"15-day slice {f_start}..{f_end}: {partialmonth_results.totalCount}")

                            for repo in partialmonth_results:
                                all_repos[repo.full_name] = repo

    return all_repos

**Export the results to CSVs file**

In [26]:
def write_repos_to_csv(all_repos, output_file):
    """
    Write GitHub repository data to a CSV file.

    Args:
        all_repos (dict): Dictionary of {repo.full_name: repo} returned by mine_repositories().
        output_file (str): Output CSV file path.
    """
    with open(output_file, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        # Header
        writer.writerow([
            "full_name",
            "description",
            "stars",
            "forks",
            "language",
            "url",
            "year",
            "last_updated",
            "topics"
        ])

        # Rows
        for name, repo in all_repos.items():
            year = repo.pushed_at.year if repo.pushed_at else ""
            last_updated = repo.pushed_at.isoformat() if repo.pushed_at else ""

            # Handle 404
            try:
                topics = ", ".join(repo.get_topics())
            except Exception:
                topics = ""

            writer.writerow([
                repo.full_name,
                repo.description or "",
                repo.stargazers_count,
                repo.forks_count,
                repo.language or "",
                repo.html_url,
                year,
                last_updated,
                topics
            ])
    print(f"CSV file created: {output_file}")

**Results**

*   CSV file 1: Agent Repositories in Python language
*   CSV file 2: Agent Repositories in other languages not Python



In [11]:
all_repos = mine_repositories(
    g=g,
    llm_keywords=llm_keywords,
    agent_keywords=agent_keywords,
    base_filters=base_filters,
    start_year=2022,
    end_year=2025
)


Running query: LLM agent language:Python fork:false archived:false created:2022-01-01..2022-12-31
Year slice 2022: 12

Running query: LLM agent language:Python fork:false archived:false created:2023-01-01..2023-12-31
Year slice 2023: 509

Running query: LLM agent language:Python fork:false archived:false created:2024-01-01..2024-12-31
Year slice 2024: 1000
Semi-yearly slice 2024-01-01..2024-06-30: 772
Semi-yearly slice 2024-07-01..2024-12-31: 1000
Quarterly slice 2024-07-01..2024-09-30: 527
Quarterly slice 2024-10-01..2024-12-31: 771

Running query: LLM agent language:Python fork:false archived:false created:2025-01-01..2025-12-31
Year slice 2025: 1000
Semi-yearly slice 2025-01-01..2025-06-30: 1000
Quarterly slice 2025-01-01..2025-03-31: 1000
Monthly slice 2025-01-01..2025-01-31: 369
Monthly slice 2025-02-01..2025-02-28: 615
Monthly slice 2025-03-01..2025-03-31: 531
Quarterly slice 2025-04-01..2025-06-30: 1000
Monthly slice 2025-04-01..2025-04-30: 693
Monthly slice 2025-05-01..2025-05

In [12]:
print(f"Collected repos so far: {len(all_repos)}")

Collected repos so far: 13340


In [13]:
output_file="github_agent_repos_python.csv"
write_repos_to_csv(all_repos, output_file)

CSV file created: github_agent_repos_python.csv


In [15]:
files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
filters = "-language:Python fork:false archived:false"

In [14]:
all_repos_notPython = mine_repositories(
    g=g,
    llm_keywords=llm_keywords,
    agent_keywords=agent_keywords,
    base_filters=filters,
    start_year=2022,
    end_year=2025
)


Running query: LLM agent -language:Python fork:false archived:false created:2022-01-01..2022-12-31
Year slice 2022: 12

Running query: LLM agent -language:Python fork:false archived:false created:2023-01-01..2023-12-31
Year slice 2023: 463

Running query: LLM agent -language:Python fork:false archived:false created:2024-01-01..2024-12-31
Year slice 2024: 1000
Semi-yearly slice 2024-01-01..2024-06-30: 700
Semi-yearly slice 2024-07-01..2024-12-31: 1000
Quarterly slice 2024-07-01..2024-09-30: 481
Quarterly slice 2024-10-01..2024-12-31: 639

Running query: LLM agent -language:Python fork:false archived:false created:2025-01-01..2025-12-31
Year slice 2025: 1000
Semi-yearly slice 2025-01-01..2025-06-30: 1000
Quarterly slice 2025-01-01..2025-03-31: 1000
Monthly slice 2025-01-01..2025-01-31: 358
Monthly slice 2025-02-01..2025-02-28: 426
Monthly slice 2025-03-01..2025-03-31: 449
Quarterly slice 2025-04-01..2025-06-30: 1000
Monthly slice 2025-04-01..2025-04-30: 517
Monthly slice 2025-05-01..202

In [28]:
print(f"Collected repos so far: {len(all_repos_notPython)}")

Collected repos so far: 10774


In [29]:
output_file="github_agent_repos_notpython.csv"
write_repos_to_csv(all_repos_notPython, output_file)
files.download(output_file)

CSV file created: github_agent_repos_notpython.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Checking sample of repo**

In [31]:
repo = g.get_repo("MLSysOps/MLE-agent")

In [32]:
repo.get_topics()

['agent', 'ai', 'llm', 'ml', 'mle', 'mlops']